# Model Selection and Hyperparameter Tuning with PetFinder Adoption Data

**Student Version**

**Shanshan (Shirley) Liu, Ph.D.**

**Summer Academy Afternoon Session, Part 2**  
**Time:** 3:45–5:00 PM  
**Main question:** *How do we choose and tune a model without overfitting to the test set?*

This notebook continues the PetFinder binary classification task from Part 1. Here we focus on model comparison, cross-validation, hyperparameter tuning, final test-set evaluation, and Random Forest feature importance.

The key workflow rule for this notebook is:

> Use the training set for model comparison and hyperparameter tuning. Save the test set for the final evaluation only.


## 1. Setup (recap from Part 1)

Part 1 built all of this and explained the reasoning. We reproduce it here in condensed form so the notebook runs end to end. If any step is unfamiliar, see Part 1 for the full walkthrough.

We reuse, unchanged from Part 1:

- the binary target `FastAdoption` (adopted on the same day or within the first week),
- the same numeric and categorical feature lists,
- an 80/20 stratified train/test split, and
- the same preprocessing pipeline (impute, then scale or one-hot encode, inside a `ColumnTransformer`).

The one rule that drives this notebook: **compare and tune on the training set, and touch the test set only for the final evaluation.**

⚠️ **Check that `petfinder.csv` is in your working directory before you continue.**

If you are running in Colab, note that your upload from Part 1 did **not** carry over: each Colab notebook gets its own runtime with its own files. Click the folder icon in the left sidebar to open the file browser, and if `petfinder.csv` is not listed, upload it with the upload icon (the page with an upward arrow). Colab also deletes uploaded files when the runtime disconnects or restarts.

Run the next cell to confirm the file is available.

In [ ]:
import os

if os.path.exists("petfinder.csv"):
    print("petfinder.csv found. Ready to go!")
else:
    print("petfinder.csv NOT found.")
    print("Upload it using the file browser (folder icon in the left sidebar in Colab), then rerun this cell.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_validate, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (
    accuracy_score,
    ConfusionMatrixDisplay,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    make_scorer,
)

RANDOM_STATE = 42
pd.set_option("display.max_columns", 100)

In [ ]:
# Recap from Part 1 -- load the data, build the binary target, and pick the feature set.
url = "https://raw.githubusercontent.com/elleobrien/MIDAS_summer_academy_student/main/day03/petfinder.csv"
raw_url = "https://raw.githubusercontent.com/elleobrien/MIDAS_summer_academy_student/main/day03/petfinder.csv"
df = pd.read_csv(url)

df = raw_df.copy()
df["FastAdoption"] = (df["AdoptionSpeed"] <= 1).astype(int)

numeric_features = [
    "Age", "Quantity", "Fee", "VideoAmt", "PhotoAmt"
]
categorical_features = [
    "Type", "Gender", "MaturitySize", "FurLength",
    "Vaccinated", "Dewormed", "Sterilized", "Health",
    "Breed1", "Color1", "State"
]
target = "FastAdoption"

X = df[numeric_features + categorical_features]
y = df[target]

# Recap from Part 1 -- 80/20 stratified split.
# Hold X_test / y_test out until the final evaluation; do all comparison and tuning on X_train.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

In [ ]:
# Recap from Part 1 -- preprocessing pipeline:
#   numeric features    -> median impute + StandardScaler
#   categorical features -> most-frequent impute + OneHotEncoder
# combined in a ColumnTransformer so it refits inside every CV fold.

# Preprocessing steps for numerical features:
# 1. Fill missing values with the median
# 2. Standardize features so they are on a similar scale
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

# Preprocessing steps for categorical features:
# 1. Fill missing values with the most frequent category
# 2. Convert categories into one-hot encoded numerical columns
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

# Apply the correct preprocessing pipeline to each group of columns
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

## 2. Compare against a dummy baseline

A dummy baseline tells us how well we could do with a very simple rule, such as always predicting the majority class.

In the training set, the most common class is Slow adoption. Therefore, this baseline predicts every pet in the training set as Slow adoption.

We calculate this baseline on the training set only. The held-out test set stays untouched until the final evaluation.

In [ ]:
majority_class = y_train.value_counts().idxmax()
dummy_pred = pd.Series(majority_class, index=y_train.index)

dummy_metrics = pd.DataFrame([{
    "Model": "Dummy majority-class baseline",
    "Training Accuracy": accuracy_score(y_train, dummy_pred),
    "Training Precision": precision_score(y_train, dummy_pred, zero_division=0),
    "Training Recall": recall_score(y_train, dummy_pred, zero_division=0),
    "Training F1": f1_score(y_train, dummy_pred, zero_division=0),
}])

dummy_metrics.round(3)

The dummy baseline gets training accuracy around the majority-class rate.

However, it is not useful for identifying fast-adoption pets.

Because it always predicts `Slow adoption`:

- It identifies **zero** fast-adoption pets.
- Its precision, recall, and F1 score for `Fast adoption` are all **0**.

This shows why accuracy can be misleading when the target classes are imbalanced.

> **Aside:** scikit-learn has a built-in version of this baseline: `DummyClassifier(strategy="most_frequent")` from `sklearn.dummy`. We computed the baseline manually here so the logic is visible, but in your own projects `DummyClassifier` is the standard tool.

<details>
<summary><b>Discussion questions</b> (optional — click to expand)</summary>

1. What does this baseline predict for every pet?
2. Why does the dummy baseline have relatively high accuracy?
3. Why are precision, recall, and F1 score for fast adoption all zero?
4. Why should we compare against a simple baseline before choosing a model?
5. Why do we calculate the baseline on the training set instead of the test set?

</details>

## 3. Define candidate model families

The goal is not to learn the theory of every model today. The goal is to compare several reasonable model choices on the same dataset and the same prediction target.

We will compare:

- `LogisticRegression`: a simple and interpretable linear classifier
- `KNeighborsClassifier`: a distance-based classifier
- `RandomForestClassifier`: an ensemble of decision trees
- `MLPClassifier`: a simple neural-network classifier from scikit-learn

All models use the same preprocessing pipeline, so the comparison is more consistent.

One caveat: the models do **not** all handle class imbalance the same way. `LogisticRegression` and `RandomForestClassifier` use `class_weight="balanced"`, but `KNeighborsClassifier` and `MLPClassifier` have no equivalent setting here. So a lower F1 for KNN or MLP partly reflects that they are not adjusting for imbalance, not only the model family itself. This is itself a useful lesson: a fair comparison means being explicit about what each model is and is not doing.


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "KNN": KNeighborsClassifier(n_neighbors=15),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        random_state=RANDOM_STATE,
        class_weight="balanced",
    ),
    # This is included only as one more scikit-learn classifier, not as a neural-network deep dive.
    "MLP Classifier": MLPClassifier(
        hidden_layer_sizes=(50,),
        max_iter=500,
        random_state=RANDOM_STATE,
    ),
}

list(models.keys())

📝 **Practice: Add one more model**

Use the [scikit-learn supervised learning guide](https://scikit-learn.org/stable/supervised_learning.html) to find one more **classification** model to try.

The docs include both classification and regression estimators. For this task, choose a classifier because `FastAdoption` is a class label, not a continuous number.

Task:

1. Find an estimator that is meant for classification.
2. If the classifier is not already imported, add its import to the first code cell.
3. Add it to the `models` dictionary below.
4. Rerun the cross-validation cell.
5. Compare its precision, recall, and F1 score with the other models.

If an estimator does not work, check whether it is a regression estimator or whether it needs different input settings.

In [ ]:
# TODO: Add one more classifier to the models dictionary here.
# Hint: look for a classifier in the scikit-learn supervised learning docs.
# Avoid regression estimators for this task because FastAdoption is a class label.

# Example structure:
# models["My classifier name"] = SomeClassifier(...)

list(models.keys())

## 4. Cross-validation for model selection

A single train/test split can be noisy. Cross-validation gives a more stable estimate of model performance using only the training set.

In 5-fold cross-validation, the training set is split into 5 smaller parts, called folds.

The model is trained 5 times:

1. Train on 4 folds, validate on the remaining fold
2. Repeat this process until each fold has been used as the validation fold once
3. Average the validation scores across the 5 runs

Importantly, we only use the **training set** for cross-validation. The test set is still held out for final evaluation.


In [ ]:
cv_scoring = {
    "accuracy": "accuracy",
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": make_scorer(recall_score, zero_division=0),
    "f1": make_scorer(f1_score, zero_division=0),
}

cv_results = []

for name, estimator in models.items():
    clf = Pipeline(steps=[
        ("preprocess", preprocessor),
        ("model", estimator),
    ])

    scores = cross_validate(
        clf,
        X_train,
        y_train,
        cv=5,
        scoring=cv_scoring,
        n_jobs=-1,
    )

    cv_results.append({
        "Model": name,
        "CV Accuracy Mean": scores["test_accuracy"].mean(),
        "CV Precision Mean": scores["test_precision"].mean(),
        "CV Recall Mean": scores["test_recall"].mean(),
        "CV F1 Mean": scores["test_f1"].mean(),
        "CV F1 Std": scores["test_f1"].std(),
    })

cv_results_df = pd.DataFrame(cv_results).sort_values("CV F1 Mean", ascending=False)
cv_results_df.round(3)

> **How to read the cross-validation results**

The table shows the average cross-validation performance for each model.

We report:

- `CV Accuracy Mean`: average accuracy across the 5 validation folds
- `CV Precision Mean`: average precision across the 5 validation folds
- `CV Recall Mean`: average recall across the 5 validation folds
- `CV F1 Mean`: average F1 score across the 5 validation folds
- `CV F1 Std`: how much the F1 score varies across folds

A higher mean score usually indicates better performance. A smaller standard deviation means the model performance is more stable across different folds.

Use this table to choose a promising model family without touching the test set. Because the target is imbalanced, focus on F1, precision, and recall instead of accuracy alone.

<details>
<summary><b>Discussion questions</b> (optional — click to expand)</summary>

1. Which model has the highest cross-validation F1 score?
2. Which model has the highest cross-validation accuracy?
3. Why is the model with the highest accuracy not necessarily the best model?
4. Which model would you choose based on cross-validation results?
5. Why do we keep the test set separate instead of using it during cross-validation?

</details>

## 5. Hyperparameter tuning with GridSearchCV

After comparing several models with cross-validation, Random Forest may look like one of the strongest options for this task.

Next, we tune the Random Forest model family. This is a practical two-stage workflow:

1. Use cross-validation to identify a promising model family.
2. Use `GridSearchCV` to tune hyperparameters within that family.

This workflow is common, but it does **not** guarantee that we have found the global best model across all possible model families and hyperparameters. A tuned Logistic Regression, KNN, or another model might outperform a tuned Random Forest.

For a broader search, we can also put multiple model families and their hyperparameter grids into one `GridSearchCV`; an optional example is shown below.

### Random Forest hyperparameters

- `n_estimators`: number of trees in the forest
- `max_depth`: maximum depth of each tree
- `min_samples_leaf`: minimum number of samples required in a leaf node

For each combination, `GridSearchCV` uses 5-fold cross-validation on the training set and evaluates the model using F1 score.

**Heads-up:** this grid search fits 18 hyperparameter combinations across 5 folds (90 model fits in total), so the next cell may take roughly 1–2 minutes to run on Colab. It is working even if it looks idle.


In [ ]:
rf_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", RandomForestClassifier(
        random_state=RANDOM_STATE,
        class_weight="balanced",
    )),
])

param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [5, 10, None],
    "model__min_samples_leaf": [1, 5, 10],
}

grid_search = GridSearchCV(
    rf_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1,
)

grid_search.fit(X_train, y_train)

print("Best parameters:")
print(grid_search.best_params_)

print()
print("Best cross-validation F1:")
print(round(grid_search.best_score_, 3))


The best hyperparameters are selected using cross-validation on the training set.

In this run, the best Random Forest settings are:

- `model__n_estimators`: **100**
- `model__max_depth`: **None**
- `model__min_samples_leaf`: **5**

The best cross-validation F1 score is about **0.447**. This is only a very small improvement over the earlier untuned Random Forest CV F1 of about **0.445**.

These exact values can shift slightly if you run on a different scikit-learn version, because several settings here produce nearly identical CV F1 scores and the tie-breaking can change which one is selected. Always read the actual winning combination from the `best_params_` output above rather than relying on the numbers quoted here.

Hyperparameter tuning may improve the model a little, but it does not magically solve all modeling problems. The quality of the features, the difficulty of the prediction task, and the available data all matter.

### Optional: tune multiple model families in one GridSearchCV

The two-stage approach above is easier to explain in class, but it only tunes Random Forest after model-family comparison.

A more exhaustive approach is to search over multiple model families and hyperparameter settings together. This can be more principled, but it can also take longer and create a larger search space.


In [ ]:
multi_model_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", LogisticRegression(max_iter=1000)),
])

multi_model_param_grid = [
    {
        "model": [LogisticRegression(max_iter=1000, class_weight="balanced")],
        "model__C": [0.1, 1, 10],
    },
    {
        "model": [RandomForestClassifier(random_state=RANDOM_STATE, class_weight="balanced")],
        "model__n_estimators": [100, 200],
        "model__max_depth": [5, 10, None],
        "model__min_samples_leaf": [1, 5, 10],
    },
]

multi_model_grid = GridSearchCV(
    multi_model_pipeline,
    param_grid=multi_model_param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1,
)

# Uncomment the next line if you want to run the broader search.
# multi_model_grid.fit(X_train, y_train)


<details>
<summary><b>Discussion questions</b> (optional — click to expand)</summary>

1. How many Random Forest hyperparameter combinations did GridSearchCV test?
2. Why do we use cross-validation inside GridSearchCV?
3. Why might we choose F1 instead of accuracy as the scoring metric?
4. Did hyperparameter tuning greatly improve the model?
5. Why might a broader multi-model grid search find a different winner?

</details>

## 6. Final test evaluation

After model comparison and hyperparameter tuning, we now evaluate the selected model on the held-out test set.

Earlier, we used the training set and cross-validation to:

- check the training data
- compare different model types
- tune hyperparameters
- select the final model

The test set was created at the beginning but kept separate during this process. Now we inspect it for the first time to estimate how well the selected model may perform on new, unseen data.

Our final selected model is the tuned Random Forest from `GridSearchCV`.


In [ ]:
best_model = grid_search.best_estimator_

final_pred = best_model.predict(X_test)

final_metrics = {
    "Accuracy": accuracy_score(y_test, final_pred),
    "Precision": precision_score(y_test, final_pred, zero_division=0),
    "Recall": recall_score(y_test, final_pred, zero_division=0),
    "F1": f1_score(y_test, final_pred, zero_division=0),
}

pd.DataFrame([final_metrics]).round(3)


The tuned Random Forest model has now been evaluated on the held-out test set.

`X_test` and `y_test` are reserved for the final model evaluation. This makes the final estimate more honest than repeatedly checking the test set during model comparison and tuning.

In this run, the final test-set metrics are approximately:

- Accuracy: **0.651**
- Precision for fast adoption: **0.355**
- Recall for fast adoption: **0.611**
- F1 for fast adoption: **0.450**

The model finds many fast-adoption pets, but its precision is modest. In other words, many pets predicted as fast adoption are actually slow adoption. The performance is still moderate, which makes sense because pet adoption speed depends on many factors that are not fully captured by the simple tabular features used in this notebook.

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    final_pred,
    display_labels=["Slow adoption", "Fast adoption"],
)
plt.title("Confusion Matrix: Tuned Random Forest")
plt.show()

print(classification_report(
    y_test,
    final_pred,
    target_names=["Slow adoption", "Fast adoption"],
    zero_division=0,
))


<details>
<summary><b>Discussion questions</b> (optional — click to expand)</summary>

Use the confusion matrix and classification report to answer:

- How many fast-adoption pets did the model correctly identify?
- How many fast-adoption pets did the model miss?
- How many slow-adoption pets were incorrectly predicted as fast adoption?
- Is recall or precision more important for the decision you want this model to support?

</details>

## 7. Random Forest feature importance

If Random Forest is the final model, it is helpful to inspect which processed features contributed most to the model's splits.

These are **model-specific feature importance scores**, not causal effects. A high importance score does not mean changing that feature would cause faster adoption. It means the trained Random Forest used that feature often or effectively when making splits.

Because categorical variables were one-hot encoded, some feature names represent specific category indicators, such as one value of `Breed1` or `State`.


In [ ]:
feature_names = best_model.named_steps["preprocess"].get_feature_names_out()
rf_importances = best_model.named_steps["model"].feature_importances_

feature_importance_df = (
    pd.DataFrame({
        "feature": feature_names,
        "importance": rf_importances,
    })
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

top_features = feature_importance_df.head(20)
top_features


In [ ]:
plot_data = top_features.sort_values("importance")

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(plot_data["feature"], plot_data["importance"])
ax.set_xlabel("Feature importance")
ax.set_title("Top 20 Random Forest Feature Importances")
plt.tight_layout()
plt.show()


<details>
<summary><b>Feature-importance discussion questions</b> (optional — click to expand)</summary>

1. Which features appear most important in the tuned Random Forest?
2. Are the most important features easy to interpret after one-hot encoding?
3. Why should we avoid interpreting these scores as causal explanations?
4. What additional features might improve prediction performance?

</details>

---

> **Summary**

The tuned Random Forest is the final model in this notebook. It has the highest training-set CV F1 among the model families we compared, but tuning improves Random Forest only slightly: from about **0.445** CV F1 to about **0.447** CV F1.

On the held-out test set, the tuned Random Forest reaches about **0.450** F1 for fast adoption. This is better than the dummy baseline, but still moderate.

This gives us several important lessons:

1. **Model selection should use the training set**  
   We used cross-validation on `X_train` and `y_train` to compare models and tune hyperparameters.

2. **The test set should be saved for final evaluation**  
   We evaluated the final selected model on `X_test` and `y_test` only after model selection was complete.

3. **Hyperparameter tuning can help, but the gain may be small**  
   GridSearchCV can improve model performance, but it does not guarantee a large gain.

4. **Accuracy alone is not enough**  
   The dummy baseline can have high accuracy while completely failing to identify fast-adoption pets.

5. **The best model depends on the goal**  
   If we care about finding as many fast-adoption pets as possible, recall is important. If we care about making reliable fast-adoption predictions, precision is important. If we want a balance between the two, F1 score is useful.

6. **Feature importance can support interpretation**  
   Feature importance can show which inputs the Random Forest used most, but it should not be treated as causal evidence.

## Optional challenge 1: Simple feature engineering

**Goal:** test whether two simple text-derived features improve the binary fast-adoption model.

Create these features:

- `DescriptionLength`: number of characters in the pet description
- `HasName`: whether the listing has a non-missing pet name

Then add them to the numerical feature list, rebuild the preprocessing pipeline, train a classifier, and compare the result with the tuned Random Forest from the main workflow.

Questions to answer:

1. Does adding `DescriptionLength` and `HasName` improve F1 for fast adoption?
2. Which metric changes the most: precision, recall, or F1?
3. Are these features easy to explain to a shelter partner?

In [ ]:
# TODO: Optional challenge 1
# 1. Copy raw_df and recreate FastAdoption.
# 2. Create DescriptionLength from Description.
# 3. Create HasName from Name.
# 4. Add the two new columns to numeric_features.
# 5. Rebuild X, y, train/test split, and preprocessing pipeline.
# 6. Train a classifier and compare its test F1 with the tuned Random Forest above.

## Optional challenge 2: Original five-class task

**Goal:** return to the original Kaggle-style target with all five `AdoptionSpeed` categories.

Instead of predicting only `FastAdoption`, predict the original classes `0`, `1`, `2`, `3`, and `4`.

Tasks:

1. Set `y_multi = raw_df["AdoptionSpeed"]`.
2. Reuse the same tabular features for `X_multi`.
3. Make a stratified train/test split.
4. Train a multiclass classifier.
5. Use `classification_report()` to compare performance across the five classes.

Question to answer: which adoption-speed classes are hardest for the model to identify?

In [ ]:
# TODO: Optional challenge 2
# 1. Set y_multi to the original AdoptionSpeed column.
# 2. Set X_multi to the same numeric + categorical features used above.
# 3. Create a stratified train/test split.
# 4. Build and fit a multiclass classifier pipeline.
# 5. Print classification_report(y_test_m, multi_pred, zero_division=0).
# 6. Identify which adoption-speed classes are hardest to predict.

<details>
<summary><b>Wrap-up discussion</b> (optional — click to expand)</summary>

1. Would you choose the tuned Random Forest as the final model? Why or why not?
2. Is this model useful for a shelter? What kind of decision could it support?
3. Which error is more concerning in this application: missing fast-adoption pets or incorrectly predicting slow-adoption pets as fast adoption?
4. What additional features might improve prediction performance?
5. If you were presenting this model to a shelter, which metric would you emphasize?
6. How would the workflow change if you tuned multiple model families at the same time?

</details>

---

### **Key takeaway**

Model selection is not just about choosing the model with the highest accuracy. A good modeling workflow should connect the evaluation metric to the real-world decision goal, compare reasonable baselines, use cross-validation for model selection, tune hyperparameters without touching the test set, and evaluate the final selected model only once on the held-out test set.

**Well done!**